# Aggregate

Daily and 30-minute request counts, built with the functions in `src/data.py` and saved to data/ for the next steps.

In [1]:
import sys

sys.path.append("..")

import pandas as pd
from src.data import load_requests, interval_counts, daily_counts

In [2]:
df = load_requests()
df.shape

(2386138, 2)

## 30-minute counts

In [3]:
interval = interval_counts(df)
interval.shape

(52560, 4)

In [4]:
interval.head()

,mobile,online,phone,total
interval,,,,
2023-11-01 00:00:00,9.0,8.0,12.0,29.0
2023-11-01 00:30:00,7.0,18.0,10.0,35.0
2023-11-01 01:00:00,4.0,7.0,5.0,16.0
2023-11-01 01:30:00,1.0,10.0,6.0,17.0
2023-11-01 02:00:00,1.0,12.0,8.0,21.0


In [5]:
interval["total"].sum(), len(df)

(np.float64(2386138.0), 2386138)

In [6]:
interval["total"].isna().sum() / 2

np.float64(275.5)

In [7]:
zeros = interval[interval["total"] == 0]
len(zeros), (zeros.index.hour < 6).mean()

(518, np.float64(0.9247104247104247))

Runs of 2 hours or more with no requests are treated as feed gaps and left empty. That's 275.5 hours in total, including the 5 missing days. The 518 empty half-hours that are left are in shorter runs, and 92% of them fall between midnight and 6am, so those look like real quiet spells.

## Daily counts

In [8]:
daily = daily_counts(interval)
daily.shape

(1095, 9)

In [9]:
daily.head()

,mobile,online,phone,total,gap_hours,outage,weekday,month,holiday
date,,,,,,,,,
2023-11-01,412.0,1386.0,2954.0,4752.0,0.0,False,2,11,False
2023-11-02,486.0,1242.0,2901.0,4629.0,0.0,False,3,11,False
2023-11-03,331.0,946.0,2011.0,3288.0,0.0,False,4,11,False
2023-11-04,258.0,563.0,744.0,1565.0,0.0,False,5,11,False
2023-11-05,187.0,505.0,536.0,1228.0,0.0,False,6,11,False


In [10]:
daily[daily["outage"]].index

DatetimeIndex(['2024-04-11', '2024-04-12', '2024-04-13', '2024-04-14',
               '2024-07-13'],
              dtype='datetime64[us]', name='date', freq=None)

In [11]:
partial = daily[(daily["gap_hours"] > 0) & ~daily["outage"]]
partial[["total", "gap_hours"]].sort_values("gap_hours", ascending=False)

,total,gap_hours
date,,
2024-07-12,821.0,15.5
2024-07-19,988.0,12.5
2024-04-15,7398.0,9.5
2026-08-15,848.0,9.5
2024-01-18,6054.0,8.5
2025-03-16,601.0,8.5
2024-07-14,3403.0,7.5
2024-07-11,786.0,7.0
2025-10-25,2396.0,7.0


29 days lose part of the day to a gap. Some of them are among the quietest days in the data (16 Mar 2025 has 601 requests but is missing 8.5 hours), and some come back with a backlog, like 15 Apr 2024 with 7,398.

In [12]:
daily.loc[daily["holiday"], ["total", "weekday"]]

,total,weekday
date,,
2023-11-10,2860.0,4
2023-11-11,1969.0,5
2023-11-23,1397.0,3
2023-12-25,1151.0,0
2024-01-01,1472.0,0
2024-01-15,3962.0,0
2024-02-19,2391.0,0
2024-05-27,876.0,0
2024-06-19,1260.0,2


## Save

In [13]:
daily.to_parquet("../data/daily.parquet")
interval.to_parquet("../data/interval.parquet")

In [14]:
pd.read_parquet("../data/daily.parquet").shape, pd.read_parquet("../data/interval.parquet").shape

((1095, 9), (52560, 4))